# EDA - Credit Risk

Notebook inicial para explorar os dados carregados no Data Warehouse.

## Observacao

O requisito de evolucao temporal nao se aplica a este dataset, porque nao ha uma coluna de data confiavel para analisar tendencia ao longo do tempo.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(Path('..') / '.env')
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'credit_risk')
db_user = os.getenv('DB_USER', 'postgres')
db_password = os.getenv('DB_PASSWORD', '')

engine = create_engine(
    f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
)

query = '''
SELECT
    f.id_aplicacao,
    c.idade,
    c.renda_anual,
    c.posse_residencia,
    c.tempo_empregada,
    p.finalidade_emprestimo,
    p.classificacao_emprestimo,
    h.inadimplencia_historica,
    h.duracao_historico_credito,
    f.montante_emprestimo,
    f.juros_aplicado,
    f.porcentagem_renda,
    f.status_emprestimo
FROM fato_emprestimo f
JOIN dim_cliente c ON c.id_cliente = f.id_cliente
JOIN dim_perfil_emprestimo p ON p.id_perfil_emprestimo = f.id_perfil_emprestimo
JOIN dim_historico_credito h ON h.id_historico_credito = f.id_historico_credito
'''

df = pd.read_sql(query, engine)
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['renda_anual'].dropna().plot(kind='hist', bins=30, ax=axes[0], title='Distribuicao da renda anual')
axes[0].set_xlabel('renda_anual')

taxa_por_residencia = df.groupby('posse_residencia')['status_emprestimo'].mean().sort_values()
taxa_por_residencia.plot(kind='bar', ax=axes[1], title='Taxa de inadimplencia por posse de residencia')
axes[1].set_xlabel('posse_residencia')
axes[1].set_ylabel('taxa de inadimplencia')

plt.tight_layout()